#### Este notebook es un clon del 08-result_group_movie_genre_managed_table de la carpeta transformation con algunas cosas modificadas para que sea carga incremental, las cosas modificadas tienen el comentario al lado, básicamente lo que se modifica es:
1. Se agrega el includes de common_functions
2. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
3. Se agrega filtro para leer las managed table particionadas de silver-inc (las que tienen datos que cambian cada semana)
4. Se agrega la columna created_date
5. Se cambia la forma de crear la managed table en el ultimo paso, el esquema, el modo de escritura, la particion, y además se agrega un paso previo que elimine registros en caso de duplicados

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")






#### Filtramos las películas con año de lanzamiento mayores o iguales al 2015

In [0]:
movie_df = spark.read.table("movie_silver_inc.movies") \
                .filter(f"file_date = '{v_file_date}'") \
                .filter("year_release_date >= 2015")
                

In [0]:
genre_df = spark.read.table("movie_silver_inc.genres")

In [0]:
movie_genre_df = spark.read.table("movie_silver_inc.movies_genres") \
                           .filter(f"file_date = '{v_file_date}'")

#### Join "movie_genres" y "genres"

In [0]:
genres_mov_gen_df = movie_genre_df.join(genre_df,
                                    movie_genre_df.genre_id == genre_df.genre_id,
                                    "inner") \
                                    .select(genre_df.genre_name, movie_genre_df.movie_id)

#### Join "movies" y "genres_mov_gen_df"

In [0]:
results_group_movie_genre = movie_df.join(genres_mov_gen_df,
                                    movie_df.movie_id == genres_mov_gen_df.movie_id,
                                    "inner") \
                                    .select(movie_df.year_release_date, movie_df.budget, 
                                            movie_df.revenue, genres_mov_gen_df.genre_name)

#### 1-Calcular el total de presupuesto y total de ingresos de las peliculas agrupadas por el año de lanzamiento y el genero
#### 2-Hacer un ranking particionado por el año de lanzamiento ordenado por el total del presupuesto y el total de ingresos de forma ascendente

In [0]:
from pyspark.sql.functions import lit

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, asc

In [0]:
df_final = results_group_movie_genre.groupBy("year_release_date", "genre_name") \
        .agg(
            sum("budget").alias("total_budget"),
            sum("revenue").alias("total_revenue")
        ) \
        .withColumn("rank", rank().over(Window.partitionBy("year_release_date").orderBy(asc("total_budget"),asc("total_revenue")))) \
        .withColumn("created_date", lit(v_file_date))

#### Escribir datos en una managed table en el contenedor gold

In [0]:
#delete_dups("movie_gold_inc","results_group_movie_genre","created_date",f"{v_file_date}")












In [0]:
#df_final.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold_inc.results_group_movie_genre")

merge_condition = 'tgt.year_release_date = src.year_release_date and tgt.genre_name = src.genre_name and tgt.created_date = src.created_date'
merge_delta_lake(df_final,'movie_gold_inc','results_group_movie_genre',merge_condition,'created_date')


















In [0]:
%sql
SELECT created_date, COUNT(1) 
FROM movie_gold_inc.results_group_movie_genre
GROUP BY created_date